# Chapter 2 — Matrices and Linear Transformations (V4, D-led)

## 2.1 The container + the verb that differentiates (Listings 2.1-2.4, Figure 2.2)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

zoning  = pd.read_csv('data/zoning.csv')
listing = pd.read_csv('data/listing.csv')
sale    = pd.read_csv('data/sale.csv')
housing = pd.merge(zoning, listing, on='Id')
housing = pd.merge(housing, sale, on='Id').set_index('Id')
print(housing.shape)

Matplotlib is building the font cache; this may take a moment.


(1460, 80)


In [2]:
n = 1000
x = np.linspace(0, 2*np.pi, n)
h = x[1] - x[0]
D = (np.eye(n, k=1) - np.eye(n)) / h
err = np.abs(D @ np.sin(x) - np.cos(x))[:-1].max()
print(f'max |D @ sin - cos|: {err:.4f}')

max |D @ sin - cos|: 0.0031


In [3]:
plt.figure(figsize=(7,4.5))
plt.plot(x, np.sin(x), label='input: sin(x)')
plt.plot(x[:-1], (D @ np.sin(x))[:-1], label='output: D @ sin(x)')
plt.plot(x, np.cos(x), 'k--', lw=1, label='cos(x)')
plt.legend()
plt.savefig('figures/fig_matrix_derivative.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.2 Linearity check (D on a combination)

In [4]:
rng = np.random.default_rng(0)
a_, xv, yv = 2.5, rng.random(n), rng.random(n)
print('max |D(ax+y) - (aDx + Dy)|:', np.abs(D @ (a_*xv + yv) - (a_*(D@xv) + D@yv)).max())

max |D(ax+y) - (aDx + Dy)|: 1.7053025658242404e-13


## 2.3 Two views + composition (Listings 2.5-2.8)

In [5]:
def by_rows(A, x):
    return np.array([row @ x for row in A])

def by_cols(A, x):
    return sum(x[j] * A[:, j] for j in range(A.shape[1]))

A = np.array([[1, 2], [3, 4], [5, 6]])
xx = np.array([7, 8])
print('A @ x      :', A @ xx)
print('row view   :', by_rows(A, xx))
print('column view:', by_cols(A, xx))

A @ x      : [23 53 83]
row view   : [23 53 83]
column view: [23 53 83]


In [6]:
# Listing 2.7 (the two views, drawn) -> Figure 2.4
plt.figure()
a1, a2 = A[:, 0], A[:, 1]
pos = np.arange(3)
parts = [('7 * a1', 7 * a1), ('8 * a2', 8 * a2),
         ('A @ x', A @ xx)]
for k, (name, vec) in enumerate(parts):
    plt.bar(pos + 0.27 * k, vec, width=0.27, label=name)
plt.xticks(pos + 0.27, ['entry 1', 'entry 2', 'entry 3'])
plt.legend()
plt.savefig('figures/fig_two_views.png', dpi=150, bbox_inches='tight')
plt.show()

In [7]:
Df = (np.eye(n, k=1) - np.eye(n)) / h
Db = (np.eye(n) - np.eye(n, k=-1)) / h
K_composed = Db @ Df
K_stencil  = (np.eye(n, k=1) - 2*np.eye(n) + np.eye(n, k=-1)) / h**2
interior = np.abs(K_composed - K_stencil)[1:-1, :].max()
err2 = np.abs((K_composed @ np.sin(x) + np.sin(x))[1:-1]).max()
print(f'composition vs stencil, interior rows: {interior:.1e}')
print(f'max |K @ sin + sin|: {err2:.6f}')

composition vs stencil, interior rows: 0.0e+00
max |K @ sin + sin|: 0.000003


In [8]:
# Listing 2.9 (the composition, drawn) -> Figure 2.5
plt.figure()
Ks = (K_composed @ np.sin(x))[1:-1]
plt.plot(x, np.sin(x), label='input: sin(x)')
plt.plot(x[1:-1], Ks, label='output: K @ sin(x)')
plt.plot(x, -np.sin(x), 'k--', lw=1, label='-sin(x)')
plt.legend()
plt.savefig('figures/fig_k_composition.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [9]:
A3 = np.array([[1, 0, 0], [-1, 1, 0], [0, -1, 1]])
print(np.linalg.inv(A3))
b = np.array([1, 3, 5])
print('x = inv(A3) @ (1,3,5):', np.linalg.inv(A3) @ b)

[[1. 0. 0.]
 [1. 1. 0.]
 [1. 1. 1.]]
x = inv(A3) @ (1,3,5): [1. 4. 9.]


In [10]:
# Listing 2.11 (the roundtrip, drawn) -> Figure 2.6
x3 = np.array([1., 4, 9])
S3 = np.linalg.inv(A3)                 # running sums
stages = [('x', x3), ('A3 @ x', A3 @ x3),
          ('inv(A3) @ A3 @ x', S3 @ A3 @ x3)]
fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=True)
for ax, (name, v) in zip(axes, stages):
    ax.stem(v)
    ax.set_title(name)
plt.savefig('figures/fig_undo_roundtrip.png', dpi=150,
            bbox_inches='tight')
plt.show()

## 2.4 The stretch and the shadow (Listings 2.9-2.12, Figures 2.4, 2.6)

In [11]:
# Listing 2.12 (two verbs, verified) — Figure 2.7 is TikZ in the book
t = np.linspace(0, 2*np.pi, 100)
circle = np.vstack([np.cos(t), np.sin(t)])
S = np.diag([2.0, 0.5])
u = np.array([2.0, 1.0])
P = np.outer(u, u) / (u @ u)   # projection onto u
ell = S @ circle
seg = P @ circle
off_line = np.abs(u[1]*seg[0] - u[0]*seg[1]).max()
print('ellipse half-axes:', ell[0].max(), ell[1].max())
print('projected circle off the u-line by:', off_line)

ellipse half-axes: 2.0 0.49993706383693753
projected circle off the u-line by: 0.0


In [12]:
print('P @ P == P?  max diff:', np.abs(P @ P - P).max())
v = np.array([1.0, 2.0])
print('residual . u =', (v - P @ v) @ u)

P @ P == P?  max diff: 1.1102230246251565e-16
residual . u = -2.220446049250313e-16


In [13]:
def arrow(vec, color, label):
    plt.quiver(0, 0, vec[0], vec[1], angles='xy', scale_units='xy',
               scale=1, color=color, label=label)

plt.figure(figsize=(6,6))
arrow(v, 'tab:blue', 'v')
arrow(P @ v, 'tab:green', 'Pv')
plt.plot(*zip(v, P @ v), 'k--', lw=1)
plt.xlim(-0.5,2.5); plt.ylim(-0.5,2.5)
plt.gca().set_aspect('equal'); plt.grid(True, alpha=0.3); plt.legend()
plt.savefig('figures/fig_projection.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.6 Standardization (Listings 2.13-2.14, Figure 2.7)

In [14]:
X = housing.select_dtypes(include='number').dropna(axis=1)
X = X.drop(columns=['SalePrice'], errors='ignore')
mu = X.mean().to_numpy()
sigma = X.std(ddof=0).to_numpy()
Z = (X.to_numpy(float) - mu) / sigma
print('columns:', X.shape[1])
print('column means after:', np.abs(Z.mean(axis=0)).max())
print('column stds after :', np.abs(Z.std(axis=0) - 1).max())

columns: 33
column means after: 3.567435540277722e-14
column stds after : 2.220446049250313e-16


In [15]:
fig, (raw, std) = plt.subplots(1, 2, figsize=(10, 4))
for col in ['LotArea', 'GrLivArea', 'OverallQual', 'YearBuilt']:
    raw.hist(X[col], bins=40, alpha=0.5, label=col)
    std.hist(Z[:, list(X.columns).index(col)], bins=40, alpha=0.5)
raw.legend(); raw.set_title('raw'); std.set_title('standardized')
plt.savefig('figures/fig_standardization.png', dpi=150, bbox_inches='tight')
plt.show()